<!-- CV-PATCHED -->

<a href="https://colab.research.google.com/github/flipiwolker-alt/cv-video-analytics/blob/main/notebooks/PZ_3_OCR_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ПЗ 3 — OCR субтитров из кадров

Читаем кадры из Drive, вырезаем нижнюю полосу с субтитрами, распознаём текст через EasyOCR.

In [ ]:
!pip install -q easyocr opencv-python-headless pandas tqdm 'numpy<2'


In [ ]:
# === Общий setup (Colab + локальный fallback) ===
import os, sys, subprocess, warnings, shutil
warnings.filterwarnings('ignore', category=SyntaxWarning)
warnings.filterwarnings('ignore', category=UserWarning)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = IN_COLAB and os.environ.get('CV_USE_DRIVE', '1') == '1'

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        BASE = '/content/drive/MyDrive/cv-frames'
    except Exception as e:
        print(f'[setup] drive недоступен ({e}) — пишем в /content/cv-frames')
        BASE = '/content/cv-frames'
elif IN_COLAB:
    BASE = '/content/cv-frames'
else:
    BASE = str((os.path.expanduser('~/cv-frames')))

BASE_DRIVE  = BASE
VIDEO_DIR   = f'{BASE}/видио'
FRAMES_DIR  = f'{BASE}/кадры'
RESULTS_DIR = f'{BASE}/результаты'
for d in (VIDEO_DIR, FRAMES_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)
print(f'[setup] BASE={BASE}')

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
if not frames:
    raise SystemExit(f'нет кадров в {FRAMES_DIR}. Сначала запустите ПЗ 2.')
print(f'кадров: {len(frames)}')


In [ ]:
import cv2
import easyocr
import pandas as pd
from tqdm.notebook import tqdm

import torch
reader = easyocr.Reader(['ru', 'en'], gpu=torch.cuda.is_available())

SUBTITLE_FRACTION = 0.15  # нижние 15% кадра

rows = []

for fname in tqdm(frames, desc='ocr'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            rows.append({'frame': fname, 'text': text, 'conf': round(prob, 3)})

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/ocr_results.csv', index=False, encoding='utf-8-sig')
print(f'распознано строк: {len(df)}')


In [6]:
from IPython.display import display

df_unique = df[['text']].drop_duplicates().reset_index(drop=True)
df_unique.index += 1
df_unique.columns = ['субтитр']

print(f'уникальных фраз: {len(df_unique)}')
display(df_unique.head(20))


уникальных фраз: 79


,субтитр
1,We're no strangers to love
2,Любовь нам не чужда
3,Were no strangers to love
4,You know the rules and so do |
5,Ты знаешь ee правила; как и я
6,"Ты знаешь ee правила, как и я"
7,You knowthe rules and so do |
8,Afull commitment's what Im thinking of
9,настроен серьезно
10,Afull commitment's what I'm
